In [1]:
import csv
import pandas as pd

In [2]:
params = {
    'fileLocation': 'database\\winemag-data-130k-v2_clean.csv',
}

In [3]:
df = csv.reader(open(params['fileLocation'], 'r', encoding='utf-8'))
columns = next(df)
columns[0] = 'id'  # Az első oszlop neve 'id' legyen
df = pd.DataFrame(df, columns=columns)

print(df.shape)
display(df.head())

(129971, 7)


,id,country,description,points,province,region_1,region_2
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",87,Sicily & Sardinia,Etna,
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",87,Douro,,
2,2,US,"Tart and snappy, the flavors of lime flesh and...",87,Oregon,Willamette Valley,Willamette Valley
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",87,Michigan,Lake Michigan Shore,
4,4,US,"Much like the regular bottling from 2012, this...",87,Oregon,Willamette Valley,Willamette Valley


In [4]:
df_description = df[['id', 'description', 'points']]
# df_description.head(5)

df_description['group'] = pd.qcut(df_description['points'].astype(int),
                                 q=4,
                                 labels=['poor', 'below_avg', 'above_avg', 'excellent'])

# display(df_description['group'].value_counts())
display(df_description.head(5))

C:\Users\2314l\AppData\Local\Temp\ipykernel_31128\1196602354.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_description['group'] = pd.qcut(df_description['points'].astype(int),


,id,description,points,group
0,0,"Aromas include tropical fruit, broom, brimston...",87,below_avg
1,1,"This is ripe and fruity, a wine that is smooth...",87,below_avg
2,2,"Tart and snappy, the flavors of lime flesh and...",87,below_avg
3,3,"Pineapple rind, lemon pith and orange blossom ...",87,below_avg
4,4,"Much like the regular bottling from 2012, this...",87,below_avg


### Szükséges könyvtárak importálása és beállítása

In [5]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from openai import OpenAI
import json

In [6]:
# NLTK adatok letöltése
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
print("NLTK csomagok sikeresen letöltve!")

NLTK csomagok sikeresen letöltve!


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\2314l\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\2314l\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Szöveg előfeldolgozása

In [7]:
def preprocess_text(text):
    """
    Előfeldolgozza a szöveget: kisbetűsítés, írásjelek eltávolítása, tokenizálás és stopwords eltávolítása.
    """
    # Kisbetűsítés
    text = text.lower()
    # Írásjelek eltávolítása
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    # Tokenizálás
    tokens = word_tokenize(text)
    # Stopwords eltávolítása
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

# Alkalmazzuk a preprocesszálást egy új oszlopba
df_description['processed_description'] = df_description['description'].apply(preprocess_text)
display(df_description.head())

C:\Users\2314l\AppData\Local\Temp\ipykernel_31128\1608395280.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_description['processed_description'] = df_description['description'].apply(preprocess_text)


,id,description,points,group,processed_description
0,0,"Aromas include tropical fruit, broom, brimston...",87,below_avg,aromas include tropical fruit broom brimstone ...
1,1,"This is ripe and fruity, a wine that is smooth...",87,below_avg,ripe fruity wine smooth still structured firm ...
2,2,"Tart and snappy, the flavors of lime flesh and...",87,below_avg,tart snappy flavors lime flesh rind dominate g...
3,3,"Pineapple rind, lemon pith and orange blossom ...",87,below_avg,pineapple rind lemon pith orange blossom start...
4,4,"Much like the regular bottling from 2012, this...",87,below_avg,much like regular bottling 2012 comes across r...


### ChatGPT API Meghívása

In [8]:
def get_client_api(params):
    """
    Inicializálja az OpenAI klienst és tárolja a paraméterszótárban.
    """
    params['client'] = OpenAI(api_key=params['api_key'])
    return params


def get_back_the_results(prompt,params,model):
    """
    Végrehajt egy ChatGPT API hívást, és visszaadja a javasolt célállomások listáját.
    """    
    params = get_client_api(params)
    client = params['client']
    response = client.chat.completions.create(
                                model = model,
                                messages=[
                                    {"role": "system", "content": "You are a wine expert assistant."},
                                    {"role": "user", "content": prompt}
                                ],
                                temperature=0.1,
                                max_tokens=50
                            )
    message = response.choices[0].message.content
    return message

def get_a_prompt(data):
    """
    Létrehoz egy promptot a megadott leírás alapján a borkategória meghatározásához.
    """
    prompt = f"""
You are a wine expert.
Based on the following wine description, predict which category the wine belongs to.

Description:
"{data}"

The possible categories are: poor, below_avg, above_avg, excellent.

Return only the category name in a JSON format like this:
{{
  "predicted_group": "<category_name>"
}}
"""
    return prompt

In [ ]:
params = {
    'api_key': "YOUR_OPENAI_API_KEY_HERE",
    'model': "gpt-4o-mini"
}

# Kiegyensúlyozott teszthalmaz létrehozása - minden csoportból 8 minta
samples_per_group = 8
test_sample = pd.DataFrame()

for group in ['poor', 'below_avg', 'above_avg', 'excellent']:
    group_data = df_description[df_description['group'] == group].sample(n=samples_per_group, random_state=42)
    test_sample = pd.concat([test_sample, group_data])

test_sample = test_sample.reset_index(drop=True)
print(f"Teszthalmaz mérete: {len(test_sample)}")
print("Csoportok eloszlása:")
print(test_sample['group'].value_counts().sort_index())

results = []
# Végigmegyünk a teljes teszthalmazon
for index, row in test_sample.iterrows():
    prompt = get_a_prompt(row['processed_description'])
    response_text = get_back_the_results(prompt, params, params['model'])
    
    try:
        # A válasz JSON-ként való értelmezése
        response_json = json.loads(response_text)
        predicted_group = response_json.get('predicted_group')
    except (json.JSONDecodeError, AttributeError):
        predicted_group = 'error'

    results.append({
        'id': row['id'],
        'description': row['description'][:100] + '...',  # Rövidítve a megjelenítéshez
        'processed_description': row['processed_description'][:100] + '...',
        'original_group': row['group'],
        'predicted_group': predicted_group
    })
    
    # Folyamat követése
    if (index + 1) % 8 == 0:
        print(f"Feldolgozva: {index + 1}/{len(test_sample)}")

df_results = pd.DataFrame(results)
display(df_results)

Teszthalmaz mérete: 32
Csoportok eloszlása:
group
poor         8
below_avg    8
above_avg    8
excellent    8
Name: count, dtype: int64
Feldolgozva: 8/32
Feldolgozva: 8/32
Feldolgozva: 16/32
Feldolgozva: 16/32
Feldolgozva: 24/32
Feldolgozva: 24/32
Feldolgozva: 32/32
Feldolgozva: 32/32


,id,description,processed_description,original_group,predicted_group
0,87452,Fruit-punchy aromas show on the seemingly swee...,fruitpunchy aromas show seemingly sweetleaning...,poor,above_avg
1,73278,"A little mealy on the nose, with apple and mel...",little mealy nose apple melon aromas poking pa...,poor,above_avg
2,91819,"Ham sandwiches, fried chicken, potato salad an...",ham sandwiches fried chicken potato salad frui...,poor,above_avg
3,72124,This is a much more oaky version of the winery...,much oaky version winerys game ranch regular b...,poor,below_avg
4,102665,If the blend of Riesling and Sauvignon Blanc i...,blend riesling sauvignon blanc isnt jarring en...,poor,poor
5,98684,Excessively ripe flavors of raisins and prunes...,excessively ripe flavors raisins prunes mar bl...,poor,poor
6,41356,Generic berry aromas are a touch gritty. This ...,generic berry aromas touch gritty malbec feels...,poor,below_avg
7,121730,"Dark golden in color, this smells sweet—like c...",dark golden color smells sweet—like caramel to...,poor,above_avg
8,57412,"Full, ripe wine, layered with fruits and a dry...",full ripe wine layered fruits dry mineral core...,below_avg,above_avg
9,38181,"This overtly spicy wine is very intense, with ...",overtly spicy wine intense slight burn palate ...,below_avg,below_avg


In [10]:
# Eredmények kiértékelése
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Hibás predikciók kiszűrése
valid_predictions = df_results[df_results['predicted_group'] != 'error']
print(f"Sikeres predikciók: {len(valid_predictions)}/{len(df_results)}")

if len(valid_predictions) > 0:
    # Pontosság számítása
    accuracy = (valid_predictions['original_group'] == valid_predictions['predicted_group']).mean()
    print(f"\nPontosság: {accuracy:.2%}")
    
    # Konfúzió mátrix
    print("\nKonfúzió mátrix:")
    categories = ['poor', 'below_avg', 'above_avg', 'excellent']
    cm = confusion_matrix(valid_predictions['original_group'], 
                         valid_predictions['predicted_group'], 
                         labels=categories)
    
    cm_df = pd.DataFrame(cm, index=categories, columns=categories)
    print(cm_df)
    
    # Részletes jelentés
    print("\nRészletes teljesítmény jelentés:")
    print(classification_report(valid_predictions['original_group'], 
                               valid_predictions['predicted_group'], 
                               labels=categories))
    
    # Hibás előrejelzések megjelenítése
    incorrect = valid_predictions[valid_predictions['original_group'] != valid_predictions['predicted_group']]
    if len(incorrect) > 0:
        print(f"\nHibás előrejelzések ({len(incorrect)} db):")
        display(incorrect[['id', 'original_group', 'predicted_group', 'description']])

Sikeres predikciók: 32/32

Pontosság: 40.62%

Konfúzió mátrix:
           poor  below_avg  above_avg  excellent
poor          2          2          4          0
below_avg     0          1          7          0
above_avg     0          0          5          3
excellent     0          0          3          5

Részletes teljesítmény jelentés:
              precision    recall  f1-score   support

        poor       1.00      0.25      0.40         8
   below_avg       0.33      0.12      0.18         8
   above_avg       0.26      0.62      0.37         8
   excellent       0.62      0.62      0.62         8

    accuracy                           0.41        32
   macro avg       0.56      0.41      0.39        32
weighted avg       0.56      0.41      0.39        32


Hibás előrejelzések (19 db):


,id,original_group,predicted_group,description
0,87452,poor,above_avg,Fruit-punchy aromas show on the seemingly swee...
1,73278,poor,above_avg,"A little mealy on the nose, with apple and mel..."
2,91819,poor,above_avg,"Ham sandwiches, fried chicken, potato salad an..."
3,72124,poor,below_avg,This is a much more oaky version of the winery...
6,41356,poor,below_avg,Generic berry aromas are a touch gritty. This ...
7,121730,poor,above_avg,"Dark golden in color, this smells sweet—like c..."
8,57412,below_avg,above_avg,"Full, ripe wine, layered with fruits and a dry..."
10,101938,below_avg,above_avg,"This is your everyday Cab, dry and tannic, but..."
11,53300,below_avg,above_avg,This wine is just over three-quarters Cabernet...
12,107442,below_avg,above_avg,"Aromas of vanilla, black cherry, toasted cocon..."
